In [1]:
import os
import sys
from dotenv import load_dotenv

# 将上一级目录（项目根目录）加入搜索路径
sys.path.append(os.path.abspath(os.path.join('..')))

# 加载环境变量
load_dotenv()

from hello_agents import (
    HelloAgentsLLM,
    SimpleAgent, ReActAgent, ReflectionAgent, PlanAndSolveAgent,
    ToolRegistry, search, calculate,
    ToolChain, ToolChainManager, AsyncToolExecutor
)

In [2]:
def demo_simple_agent():
    """演示SimpleAgent - 基础对话"""
    print("\n" + "="*60)
    print("🤖 SimpleAgent 演示 - 基础对话Agent")
    print("="*60)

    # 创建LLM实例
    llm = HelloAgentsLLM(
        model=os.getenv("OPENAI_MODEL_ID", "gpt-5.1-free"),
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL")
    )

    # 创建简单Agent
    agent = SimpleAgent(
        name="助手",
        llm=llm,
        system_prompt="你是一个有用的AI助手，请用中文回答问题。"
    )

    # 测试对话
    test_questions = [
        "你好，请介绍一下自己",
        "什么是人工智能？",
        "请用一句话总结机器学习的核心思想"
    ]

    for question in test_questions:
        print(f"\n用户: {question}")
        try:
            response = agent.run(question)
            print(f"助手: {response}")
        except Exception as e:
            print(f"❌ 错误: {e}")

In [3]:
demo_simple_agent()


🤖 SimpleAgent 演示 - 基础对话Agent

用户: 你好，请介绍一下自己
助手: 你好！我是由Z.ai训练的大型语言模型。我的设计目标是理解和生成人类语言，从而协助你解决问题。无论是回答问题、提供信息、协助写作、翻译语言，还是进行日常对话，我都在这里为你提供帮助。

请问有什么我能为你做的吗？

用户: 什么是人工智能？
助手: 人工智能（Artificial Intelligence，简称AI）是一门研究如何使计算机模拟人类智能行为的科学技术。简单来说，就是让机器能够像人一样去思考、学习、解决问题。

可以从以下几个方面来理解：

**1. 核心目标：**
AI的最终目标是创造出能够执行通常需要人类智能才能完成的任务的系统。这包括视觉感知、语音识别、决策制定、语言翻译等。

**2. 主要分类：**
*   **弱人工智能：** 专注于在特定领域执行单一任务的AI。例如，语音助手、推荐算法、围棋程序等。这是目前我们生活中接触到的AI形式。
*   **强人工智能：** 指拥有与人类同等甚至超越人类智慧，能够像人一样进行通用思考、推理和解决各种复杂问题的AI。目前这种AI还主要存在于科幻作品和理论研究阶段。

**3. 关键技术：**
*   **机器学习：** 让计算机能够从大量数据中自动学习和改进，而不需要显式地编程。这是AI当前发展的核心驱动力。
*   **深度学习：** 机器学习的一个分支，它通过模拟人脑神经网络的结构，让计算机能够处理非结构化数据（如图像、声音、文本）。

**4. 应用场景：**
AI已经渗透到我们生活的方方面面，比如：
*   **智能助手：** Siri、小爱同学等。
*   **推荐系统：** 抖音、淘宝等平台的个性化推荐。
*   **自动驾驶：** 汽车的自动驾驶技术。
*   **医疗诊断：** 辅助医生进行疾病筛查和诊断。

总的来说，AI是一项旨在通过赋予机器智能来延伸和增强人类能力的技术，它正在深刻地改变我们的生活方式和社会结构。

用户: 请用一句话总结机器学习的核心思想
助手: 机器学习的核心思想是让计算机通过分析大量数据自动寻找规律，从而在没有显式编程的情况下完成特定任务或做出预测。


In [2]:
def demo_react_agent():
    """演示ReActAgent - 推理与行动结合"""
    print("\n" + "="*60)
    print("🔧 ReActAgent 演示 - 推理与行动结合的Agent")
    print("="*60)

    # 创建LLM实例
    llm = HelloAgentsLLM(
        model=os.getenv("OPENAI_MODEL_ID", "coding-glm-4.6-free"),
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL")
    )

    # 创建工具注册表
    tool_registry = ToolRegistry()

    # 注册工具
    tool_registry.register_function(
        name="search",
        description="一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。",
        func=search
    )

    tool_registry.register_function(
        name="calculate",
        description="执行数学计算。支持基本运算、数学函数等。例如：2+3*4, sqrt(16), sin(pi/2)等。",
        func=calculate
    )

    # 1. 默认配置演示
    print("\n--- 默认配置 ---")
    default_agent = ReActAgent(
        name="通用助手",
        llm=llm,
        tool_registry=tool_registry,
        max_steps=3
    )

    task1 = "计算 15 * 23 + 45 的结果"
    print(f"\n🎯 任务: {task1}")
    try:
        response = default_agent.run(task1)
        print(f"\n✅ 默认配置结果: {response}")
    except Exception as e:
        print(f"❌ 错误: {e}")

    # 2. 自定义配置演示 - 研究助手
    print("\n--- 自定义配置：研究助手 ---")
    research_prompt = """
你是一个专业的研究助手，擅长信息收集和分析。

可用工具如下：
{tools}

请按照以下格式进行研究：

Thought: 分析问题，确定需要什么信息，制定研究策略。
Action: 选择合适的工具获取信息，格式为：
- `{{tool_name}}[{{tool_input}}]`：调用工具获取信息。
- `Finish[研究结论]`：当你有足够信息得出结论时。

研究问题：{question}
已完成的研究：{history}
"""

    research_agent = ReActAgent(
        name="研究助手",
        llm=llm,
        tool_registry=tool_registry,
        custom_prompt=research_prompt,
        max_steps=3
    )

    task2 = "搜索一下最新的人工智能发展趋势"
    print(f"\n🎯 任务: {task2}")
    try:
        response = research_agent.run(task2)
        print(f"\n✅ 研究助手结果: {response}")
    except Exception as e:
        print(f"❌ 错误: {e}")

In [3]:
demo_react_agent()


🔧 ReActAgent 演示 - 推理与行动结合的Agent
✅ 工具 'search' 已注册。
✅ 工具 'calculate' 已注册。

--- 默认配置 ---

🎯 任务: 计算 15 * 23 + 45 的结果

🤖 通用助手 开始处理问题: 计算 15 * 23 + 45 的结果

--- 第 1 步 ---
🤔 思考: 用户要求计算 15 * 23 + 45，这是一个简单的数学计算问题，我需要使用 calculate 工具来完成。
🎬 行动: calculate[15 * 23 + 45]
🧮 正在计算: 15 * 23 + 45
✅ 计算结果: 390
👀 观察: 390

--- 第 2 步 ---
🤔 思考: 我已经通过calculate工具计算了表达式 15 * 23 + 45，得到了结果390。验证一下：15 × 23 = 345，345 + 45 = 390，结果正确。现在可以给出最终答案。
🎉 最终答案: 15 * 23 + 45 = 390

✅ 默认配置结果: 15 * 23 + 45 = 390

--- 自定义配置：研究助手 ---

🎯 任务: 搜索一下最新的人工智能发展趋势

🤖 研究助手 开始处理问题: 搜索一下最新的人工智能发展趋势

--- 第 1 步 ---
🤔 思考: 用户询问最新的人工智能发展趋势。由于AI领域更新速度极快，涉及大模型、生成式AI、监管政策等多个方面，我需要通过搜索引擎获取最新的行业报告、新闻报道和技术突破，以确保信息的时效性和准确性。

--- 第 2 步 ---
🤔 思考: 用户询问最新的人工智能发展趋势，这是一个需要最新数据的动态问题。我的知识库可能无法涵盖最新的突破，因此需要使用 `search` 工具来获取2024年至2025年关于AI领域的最新报告、技术突破（如多模态、大模型）、行业应用及监管政策等方面的信息。

--- 第 3 步 ---
🤔 思考: 用户想了解最新的人工智能发展趋势。这是一个关于时事和前沿科技的动态问题，超出了我的静态知识库范围。我需要使用搜索引擎来获取2024年至2025年期间AI领域的最新动态、技术突破和应用趋势。
🎬 行动: search[最新人工智能发展趋势 2024 2025]
✅ Tavily搜索引擎已初始化
✅ SerpApi搜索引擎已初始化
🔧

In [3]:
def demo_reflection_agent():
    """演示ReflectionAgent - 自我反思与迭代优化"""
    print("\n" + "="*60)
    print("🔄 ReflectionAgent 演示 - 自我反思与迭代优化的Agent")
    print("="*60)

    # 创建LLM实例
    llm = HelloAgentsLLM()

    # 1. 默认配置演示
    print("\n--- 默认配置 ---")
    default_agent = ReflectionAgent(
        name="通用助手",
        llm=llm,
        max_iterations=2
    )

    task1 = "解释什么是递归算法，并给出一个简单的例子"
    print(f"\n🎯 任务: {task1}")
    try:
        response = default_agent.run(task1)
        print(f"\n✅ 默认配置结果:\n{response}")
    except Exception as e:
        print(f"❌ 错误: {e}")

    # 2. 自定义配置演示 - 代码生成专家
    print("\n--- 自定义配置：代码生成专家 ---")
    code_prompts = {
        "initial": """
你是一位资深的程序员。请根据以下要求编写代码：

要求: {task}

请提供完整的代码实现，包含必要的注释和文档。
""",
        "reflect": """
你是一位严格的代码评审专家。请审查以下代码：

# 原始任务: {task}
# 待审查的代码: {content}

请分析代码的质量，包括算法效率、可读性、错误处理等。
如果代码质量良好，请回答"无需改进"。否则请提出具体的改进建议。
""",
        "refine": """
请根据代码评审意见优化你的代码：

# 原始任务: {task}
# 上一轮代码: {last_attempt}
# 评审意见: {feedback}

请提供优化后的代码。
"""
    }

    code_agent = ReflectionAgent(
        name="代码专家",
        llm=llm,
        custom_prompts=code_prompts,
        max_iterations=2
    )

    task2 = "编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。"
    print(f"\n🎯 任务: {task2}")
    try:
        response = code_agent.run(task2)
        print(f"\n✅ 代码专家结果:\n{response}")
    except Exception as e:
        print(f"❌ 错误: {e}")

In [4]:
demo_reflection_agent()


🔄 ReflectionAgent 演示 - 自我反思与迭代优化的Agent

--- 默认配置 ---

🎯 任务: 解释什么是递归算法，并给出一个简单的例子

🤖 通用助手 开始处理任务: 解释什么是递归算法，并给出一个简单的例子

--- 正在进行初始尝试 ---
📝 记忆已更新，新增一条 'execution' 记录。

--- 第 1/2 轮迭代 ---

-> 正在进行反思...
📝 记忆已更新，新增一条 'reflection' 记录。

✅ 反思认为结果已无需改进，任务完成。

--- 任务完成 ---
最终结果:
好的，我将为您解释什么是递归算法，并提供一个简单的例子。

### 一、什么是递归算法？

递归算法是一种在函数或过程中 **调用自身** 来解决问题的方法。其核心思想是：将一个复杂问题分解为 **规模更小、但结构相同** 的子问题，直到子问题足够简单可以直接求解（基本情况），然后逐层返回结果，最终解决原问题。

递归算法必须包含两个关键部分：
1.  **基本情况（Base Case）**：也称为“递归出口”，即问题可以直接求解的最简单情况。它确保递归不会无限进行。
2.  **递归情况（Recursive Case）**：将问题分解为更小的子问题，并调用自身来解决这些子问题。

**可以想象为“俄罗斯套娃”**：你打开一个大娃娃，里面有一个小一点的娃娃，再打开，里面还有更小的……直到你打开一个最小的娃娃，里面没有更小的了（基本情况）。然后你逐层把信息往外传递。

---

### 二、简单例子：计算阶乘

阶乘（Factorial）是递归最经典的例子。n的阶乘记为 `n!`，定义为：
- `0! = 1`（基本情况）
- `n! = n × (n-1)!`（递归情况）

#### 1. 递归算法描述
- **基本情况**：如果 `n = 0`，返回 `1`。
- **递归情况**：返回 `n × (n-1)的阶乘`。

#### 2. Python 代码示例
```python
def factorial(n):
    # 基本情况
    if n == 0:
        return 1
    # 递归情况
    else:
        return n * factorial(n - 1)

# 测试
print

In [5]:
def demo_plan_solve_agent():
    """演示PlanAndSolveAgent - 分解规划与逐步执行"""
    print("\n" + "="*60)
    print("📋 PlanAndSolveAgent 演示 - 分解规划与逐步执行的Agent")
    print("="*60)

    # 创建LLM实例
    llm = HelloAgentsLLM()

    # 1. 默认配置演示
    print("\n--- 默认配置 ---")
    default_agent = PlanAndSolveAgent(
        name="通用助手",
        llm=llm
    )

    task1 = "如何学习Python编程？请制定一个详细的学习计划。"
    print(f"\n🎯 任务: {task1}")
    try:
        response = default_agent.run(task1)
        print(f"\n✅ 默认配置结果:\n{response}")
    except Exception as e:
        print(f"❌ 错误: {e}")

    # 2. 自定义配置演示 - 数学问题专家
    print("\n--- 自定义配置：数学问题专家 ---")
    math_prompts = {
        "planner": """
你是一个数学问题分解专家。请将以下数学问题分解为清晰的计算步骤。
每个步骤应该是一个具体的数学运算或逻辑推理。

数学问题: {question}

请按以下格式输出计算计划:
```python
["步骤1: 具体计算", "步骤2: 具体计算", ...]
```
""",
        "executor": """
你是一个数学计算专家。请严格按照计划执行数学计算。

# 原始问题: {question}
# 计算计划: {plan}
# 已完成的计算: {history}
# 当前计算步骤: {current_step}

请执行当前步骤的计算，只输出计算结果:
"""
    }

    math_agent = PlanAndSolveAgent(
        name="数学专家",
        llm=llm,
        custom_prompts=math_prompts
    )

    task2 = "一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？"
    print(f"\n🎯 任务: {task2}")
    try:
        response = math_agent.run(task2)
        print(f"\n✅ 数学专家结果:\n{response}")
    except Exception as e:
        print(f"❌ 错误: {e}")

In [6]:
demo_plan_solve_agent()


📋 PlanAndSolveAgent 演示 - 分解规划与逐步执行的Agent

--- 默认配置 ---

🎯 任务: 如何学习Python编程？请制定一个详细的学习计划。

🤖 通用助手 开始处理问题: 如何学习Python编程？请制定一个详细的学习计划。
--- 正在生成计划 ---
✅ 计划已生成:
```python
["安装Python解释器和集成开发环境（如VS Code或PyCharm），并配置环境变量。", "学习Python基础语法，包括变量、数据类型、字符串操作和打印语句。", "掌握运算符、表达式和基本输入输出功能。", "学习条件语句（if-elif-else）和循环结构（for、while）。", "定义和使用函数，包括参数传递和返回值。", "了解模块和包的概念，学习导入标准库和第三方库。", "操作基本数据结构：列表、元组、字典和集合。", "学习面向对象编程基础，包括类、对象、继承和多态。", "处理文件读写操作，如打开、读取、写入和关闭文件。", "实现错误和异常处理，使用try-except语句。", "通过编写简单程序（如计算器、猜数字游戏）实践所学知识。", "学习调试技巧和单元测试，确保代码质量。", "探索Python标准库（如os、math、datetime）和常用第三方库（如requests、pandas）。", "选择一个感兴趣的方向（如Web开发、数据分析、自动化脚本）进行进阶学习。", "参与开源项目或完成个人项目，以巩固和扩展技能。"]
```

--- 正在执行计划 ---

-> 正在执行步骤 1/15: 安装Python解释器和集成开发环境（如VS Code或PyCharm），并配置环境变量。
✅ 步骤 1 已完成，结果: 安装Python解释器和集成开发环境（如VS Code或PyCharm），并配置环境变量。

**1. 安装Python解释器**
*   访问Python官网（https://www.python.org/downloads/）下载最新稳定版（如Python 3.12或3.11）。
*   运行安装程序，**务必勾选“Add Python to PATH”**（或类似的“Add to environment variables”选项），这将自动配置环境变量。
*   选择“In

KeyboardInterrupt: 

In [7]:
def demo_advanced_features():
    """演示高级功能：工具链和异步执行"""
    print("\n" + "="*60)
    print("🚀 高级功能演示 - 工具链和异步执行")
    print("="*60)

    # 创建工具注册表
    registry = ToolRegistry()
    registry.register_function("calculate", "数学计算工具", calculate)

    # 1. 工具链演示
    print("\n--- 工具链演示 ---")

    # 创建简单的工具链
    chain = ToolChain("demo_chain", "演示工具链")
    chain.add_step("calculate", "2 + 3", "step1")
    chain.add_step("calculate", "5 * 2", "step2")

    # 创建工具链管理器
    chain_manager = ToolChainManager(registry)
    chain_manager.register_chain(chain)

    # 执行工具链
    print("🔗 执行工具链...")
    result = chain_manager.execute_chain("demo_chain", "开始")
    print(f"✅ 工具链结果: {result}")

In [8]:
demo_advanced_features()


🚀 高级功能演示 - 工具链和异步执行
✅ 工具 'calculate' 已注册。

--- 工具链演示 ---
✅ 工具链 'demo_chain' 添加步骤: calculate
✅ 工具链 'demo_chain' 添加步骤: calculate
✅ 工具链 'demo_chain' 已注册
🔗 执行工具链...
🚀 开始执行工具链: demo_chain
📝 执行步骤 1/2: calculate
🧮 正在计算: 2 + 3
✅ 计算结果: 5
✅ 步骤 1 完成
📝 执行步骤 2/2: calculate
🧮 正在计算: 5 * 2
✅ 计算结果: 10
✅ 步骤 2 完成
🎉 工具链 'demo_chain' 执行完成
✅ 工具链结果: 10


In [9]:
registry = ToolRegistry()
registry.register_function("calculate", "数学计算工具", calculate)

import asyncio

async def async_demo():
    # 创建异步执行器
    executor = AsyncToolExecutor(registry, max_workers=2)

    # 定义并行任务
    tasks = [
        {"tool_name": "calculate", "input_data": "10 + 5"},
        {"tool_name": "calculate", "input_data": "20 * 3"},
        {"tool_name": "calculate", "input_data": "100 / 4"},
    ]

    print("⚡ 并行执行多个计算任务...")
    results = await executor.execute_tools_parallel(tasks)

    print("📊 并行执行结果:")
    for result in results:
        status = "✅" if result["status"] == "success" else "❌"
        print(f"{status} {result['input_data']} = {result['result']}")

    executor.close()

await async_demo()

✅ 工具 'calculate' 已注册。
⚡ 并行执行多个计算任务...
🚀 开始并行执行 3 个工具任务
📝 创建任务 1: calculate
📝 创建任务 2: calculate
📝 创建任务 3: calculate
🧮 正在计算: 10 + 5
✅ 计算结果: 15
✅ 任务 1 完成: calculate
🧮 正在计算: 20 * 3
✅ 计算结果: 60
✅ 任务 2 完成: calculate
🧮 正在计算: 100 / 4
✅ 计算结果: 25.0
✅ 任务 3 完成: calculate
🎉 并行执行完成，成功: 3/3
📊 并行执行结果:
✅ 10 + 5 = 15
✅ 20 * 3 = 60
✅ 100 / 4 = 25.0
🔒 异步工具执行器已关闭
